In [1]:
import os
import sys
from pathlib import Path
import polars as pl
import pandas as pd
import numpy as np
import gc
from catboost import CatBoostClassifier, Pool
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, log_loss, confusion_matrix, classification_report, precision_recall_curve, balanced_accuracy_score, matthews_corrcoef, average_precision_score, ConfusionMatrixDisplay
from sklearn.model_selection import ParameterSampler
import matplotlib.pyplot as plt

current_dir = Path(__file__).resolve().parent if "__file__" in locals() else Path.cwd()
sys.path.append(str(current_dir.parent))
from utils import *
from schemas import *
from clean import *
from visualisation_utils import *

pl.Config.set_tbl_cols(-1)
os.chdir(r'E:\CVUT_BAP')
# os.chdir(r'C:\Users\adamp\Projects\CVUT_BAP')
SEED=42
PRINT = False

# 1. Načtení dat

In [2]:
lf_prohlidky = pl.scan_parquet(r"E:\CVUT_BAP\kod\data\processed\prohlidky_model.parquet")
# lf_mereni = pl.scan_parquet(r"E:\CVUT_BAP\kod\data\processed\mereni_model.parquet")
lf_mereni = pl.scan_parquet(r"E:\CVUT_BAP\kod\data\processed\mereni_model_raw.parquet")
df = lf_prohlidky.join(lf_mereni, on='CisloProtokolu', how='inner').collect()#.sample(fraction=0.1, seed=SEED)

# 2. Definice proměnných

In [3]:
id_col = 'CisloProtokolu'
target_col = 'Vysledek_Pristi'

# Identifikace sloupců
all_cols = df.columns
feature_cols = [c for c in all_cols if c not in [id_col, target_col]]

# Automatická detekce kategorických a numerických sloupců
cat_features = [c for c in feature_cols if df[c].dtype in [pl.String, pl.Boolean, pl.Categorical]]
num_features = [c for c in feature_cols if df[c].dtype in [pl.Float32, pl.Float64, pl.Int32, pl.Int8, pl.UInt32]]

# 3. Typová optimalizace (Polars) pro minimalizaci 

In [4]:
df = df.with_columns([
    pl.col(c).cast(pl.Float32) for c in num_features
]).with_columns([
    pl.col(c).cast(pl.String).fill_null("missing") for c in cat_features
]).with_columns(
    # Inverze zajistí, že 1 = Neúspěch.
    (1 - pl.col(target_col)).cast(pl.Int8).alias(target_col)
)

# 4. Konverze do Pandas 

In [5]:
# ID sloupec je vyřazen z trénovacích dat
df_pd = df.select(feature_cols + [target_col]).to_pandas()

# Odstranění Polars DataFrame z paměti
del df
_ = gc.collect()

# 5. Rozdělení dat (Train 80 %, Val 10 %, Test 10 %) se stratifikací

In [6]:
X = df_pd[feature_cols]
y = df_pd[target_col]

X_temp, X_test, y_temp, y_test = train_test_split(X, y, test_size=0.10, random_state=SEED, stratify=y)
X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size=0.1111, random_state=SEED, stratify=y_temp)

# Odstranění dočasných proměnných
del df_pd, X, y, X_temp, y_temp
_ = gc.collect()

# 6. Inicializace objektů Pool

In [7]:
# Převedení dat do interního formátu CatBoost zefektivní využití paměti na GPU
train_pool = Pool(X_train, y_train, cat_features=cat_features)
val_pool = Pool(X_val, y_val, cat_features=cat_features)
test_pool = Pool(X_test, y_test, cat_features=cat_features)

del X_train, y_train, X_val, y_val, X_test, y_test
_ = gc.collect()

# 7. Konfigurace a trénování modelu 

In [ ]:
# Definice prostoru hyperparametrů
# param_grid = {
#     'learning_rate': [0.03, 0.05, 0.1],
#     'depth': [4, 6, 8],
#     'l2_leaf_reg': [1, 3, 5]
# }
param_grid = {'learning_rate': [0.03], 'l2_leaf_reg': [3], 'depth': [8]}

# Výběr 4 náhodných kombinací pro úsporu času na starším GPU
param_list = list(ParameterSampler(param_grid, n_iter=4, random_state=SEED))

best_score = float('inf')
best_params = None
best_model = None

print("--- Start Hyperparameter Tuningu ---")
for i, params in enumerate(param_list):
    print(f"\nModel {i+1}/{len(param_list)} | Parametry: {params}")
    
    current_model = CatBoostClassifier(
        iterations=10000,              # vysoky limit, konvergenci řídí early_stopping
        task_type='CPU',#'GPU',  
        thread_count=-1,       
        #devices='0',             
        max_ctr_complexity=1,    
        eval_metric='Logloss',
        random_seed=SEED,
        early_stopping_rounds=50,
        auto_class_weights='Balanced',
        **params
    )
    
    current_model.fit(
        train_pool,
        eval_set=val_pool,
        verbose=50
    )
    
    # Vyhodnocení Logloss na validační sadě
    val_loss = current_model.get_best_score()['validation']['Logloss']
    print(f"Dosažený Logloss: {val_loss:.4f} v iteraci {current_model.get_best_iteration()}")
    
    if val_loss < best_score:
        best_score = val_loss
        best_params = params
        best_model = current_model

print(f"\n--- Konec Tuningu ---")
print(f"Nejlepší parametry: {best_params}")

# Model s nejlepšími parametry
model = best_model

e:\CVUT_BAP\.venv\Lib\site-packages\sklearn\model_selection\_search.py:324: UserWarning: The total space of parameters 1 is smaller than n_iter=4. Running 1 iterations. For exhaustive searches, use GridSearchCV.
  warnings.warn(


--- Start Hyperparameter Tuningu ---

Model 1/1 | Parametry: {'learning_rate': 0.03, 'l2_leaf_reg': 3, 'depth': 8}
0:	learn: 0.6892008	test: 0.6892472	best: 0.6892472 (0)	total: 18.5s	remaining: 2d 3h 17m 53s
50:	learn: 0.6214540	test: 0.6229461	best: 0.6229461 (50)	total: 12m 33s	remaining: 1d 16h 49m 34s
100:	learn: 0.6105864	test: 0.6123371	best: 0.6123371 (100)	total: 24m 58s	remaining: 1d 16h 48m 14s
150:	learn: 0.6063319	test: 0.6083612	best: 0.6083612 (150)	total: 37m 36s	remaining: 1d 16h 53m 29s
200:	learn: 0.6035251	test: 0.6057896	best: 0.6057896 (200)	total: 49m 45s	remaining: 1d 16h 25m 48s
250:	learn: 0.6013914	test: 0.6039400	best: 0.6039400 (250)	total: 1h 1m 34s	remaining: 1d 15h 51m 55s


# 8. Vyhodnocení na testovací sadě

In [ ]:
# Aplikace fixního prahu a vyhodnocení na TESTOVACÍ SADĚ
optimal_threshold = 0.5
y_test_proba = model.predict_proba(test_pool)[:, 1] # type: ignore
y_test_labels = test_pool.get_label()
y_test_pred = (y_test_proba >= optimal_threshold).astype(int)

# Výpočet matic (absolutní i relativní)
cm_abs = confusion_matrix(y_test_labels, y_test_pred)
cm_rel = confusion_matrix(y_test_labels, y_test_pred, normalize='true')

# Výpis ukazatelů
print(f"Použitý práh: {optimal_threshold:.1f}")
print(f"Balanced Accuracy (Test): {balanced_accuracy_score(y_test_labels, y_test_pred):.4f}")
print("\n--- Klasifikační report na TESTOVACÍ sadě (1 = Neúspěch) ---")
print(classification_report(y_test_labels, y_test_pred, target_names=['Úspěch (0)', 'Neúspěch (1)']))

# Vizualizace relativní matice záměn
fig, ax = plt.subplots(figsize=(6, 5))
disp = ConfusionMatrixDisplay(
    confusion_matrix=cm_rel, 
    display_labels=['Úspěch (0)', 'Neúspěch (1)']
)
_ = disp.plot(cmap='Blues', ax=ax, values_format='.2%')